# Machine Learning in Python - Project

Due Friday, December 4th 2026, by 4 pm.


**Students: Pierrick de Boer S3034691, Richard Mugisha S3032547, Goodness Balogun S3031177, Nuha Hamada S3026915**


This section is to import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
!pip install cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

  Using cached cartopy-0.26.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (6.2 kB)
  Using cached pyshp-3.1.9-py3-none-any.whl.metadata (79 kB)
Using cached cartopy-0.26.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (9.5 MB)
Using cached pyshp-3.1.9-py3-none-any.whl (75 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cartopy]m1/2 [cartopy]


# Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. Scottish Government)).*

*State which supervised learning question you are answering.*

*Briefly outline the approaches being used (for both the clustering and your chosen supervised learning task) and the conclusions that you are able to draw.*

We will be investigating river water levels in Scotland. https://riverlevels.uk/ Four datasets have been created, containing
information about 392 river level monitoring stations. The first three datasets contain the minimum
daily water level, average daily water level, and maximum daily water level, respectively, the fourth
dataset contains the latitude and longitude of each station’s location. Not all stations have recorded
data across the entire time period available. Your goal is to build two models to provide the Scottish
government with useful insights about river water levels.

# Exploratory Data Analysis and Feature Engineering

## Clustering

### Data structure and quality

The data comprise daily minimum, average and maximum water levels for 392 stations over 4379 days from 05/10/2014 to 08/10/2026, stored as one row per station and one column per day, together with a fourth file holding each station's coordinates. Before any cleaning, we validated the raw files, since the cleaning steps rely on assumptions about structure, identifiers and value ranges. We checked that:

- none of the files is empty,
- each level file contains the columns County, Station_Name and Observation_ID, and every other column is a date in the format dd/mm/yyyy,
- every row has a station identifier, which is the only key linking the four files,
- all four files contain exactly the same stations, so that stations can be matched by their identifier rather than by row order (the location file lists them in a different order),
- no station appears twice,
- all values are numeric,
- the number of missing values per file,
- the number of measured days per station and each station's first and last measurement date,
- the physical constraint $min \leq avg \leq max$ holds for every station-day, allowing a tolerance of 0.006 m because the minimum and maximum are rounded to two decimals,
- the coordinates lie within valid latitude and longitude ranges.

Before matching, station names and identifiers were harmonised with respect to whitespace and capitalisation. All structural checks pass: there are no duplicates, no non-numeric entries, no violations of $min \leq avg \leq max$ and no invalid coordinates. The main issues are therefore missing data, which makes the data imbalanced, and a small number of implausible values.

### Missing data and imbalance

About 30.6% of all station-days are missing. The missing entries are identical in the minimum, average and maximum files, so we assume they mark periods in which a station was not operating rather than random measurement failures. Coverage varies strongly between stations, from 452 to 4373 days (Figure 1a).

<p align="center">
  <img src="Figure1.png" width="30%">
</p>
<p align="center"><em>Figure 1a: Measured days per station in the raw data (392 stations, file order). Balanced data would form a flat block at the dashed line.</em></p>

Stations started recording between 2014 and 2024 and some stopped as early as 2021, so no single period is covered by all of them. Statistics computed over each station's own period would mix different years and make stations incomparable, which would bias any clustering.

We therefore undersample in time. All stations are restricted to a common period, from the median start to the median end date (31/03/2017 – 08/10/2026, 3,479 days), and only stations with at least 90% coverage within it are kept. This retains **301 of 392 stations**. After undersampling, all stations cover the same period with similar coverage (Figure 1b).

<p align="center">
  <img src="Figure1b.png" width="30%">
</p>
<p align="center"><em>Figure 1b: Measured days per station after undersampling to the common period (301 stations).</em></p>

The 91 dropped stations are spread across Scotland rather than concentrated in one region, so undersampling does not systematically exclude any area, which was our main concern with this approach. Within the common period, [5.8]% of values are still missing and every station has at least one gap. The gaps are scattered across stations and dates, so only [1,097] days are complete for all stations; these form the input to PCA.

### Measurement errors

Fourteen stations contain physically implausible values: isolated spikes of up to 170,213 m, or down to −190,475 m, in rivers whose typical level is below a few metres. A single such value produces a skewness close to $\sqrt{n} \approx 57$, the value expected when one observation dominates an otherwise regular series. The positive spikes lie near multiples of around 20,833, which suggests that a placeholder value from the sub-daily records entered the daily averages. These errors are critical for clustering: a single spike inflates a station's standard deviation by several orders of magnitude, so after standardisation (see below) its regular behaviour collapses to zero and only the spike remains. We set to missing every value more than 50 robust standard deviations from its station's median $(\text{median} \pm 50 \cdot 1.4826 \cdot \text{MAD})$, separately for each file. Values beyond 8 robust standard deviations ([3,188] values) are only flagged, since these are mostly genuine flood events and part of the signal we want to cluster.

### Feature engineering for clustering

Each station is represented by its series of average daily water levels on the complete days, giving a matrix with one row per station and one column per day. We use only the average level, because the minimum and maximum are almost perfectly correlated with it and add little information. Two preprocessing steps follow, both implemented as an sklearn pipeline:

1. **Standardisation per station.** Every station's series is scaled to mean 0 and standard deviation 1 over time. Gauges measure relative to different local reference levels and rivers differ greatly in size, so without this step PCA would mainly separate large from small rivers. After standardisation, stations are compared by *when* their levels rise and fall. Note that sklearn's StandardScaler scales each column (day), not each row (station), so we use a custom FunctionTransformer.
2. **Dimension reduction with PCA.** With roughly [1,100] days per station but only 301 stations, distances in the raw space are dominated by day-to-day noise. PCA keeps the directions in which many stations vary together, such as shared rain events and seasonal patterns. We retain **30 principal components**, which explain [80]% of the variance (Figure 2). Beyond this point each additional component adds very little and mostly captures station-specific noise rather than shared behaviour.

<p align="center">
  <img src="Figure3.png" width="20%">
</p>
<p align="center"><em>Figure 2: Cumulative share of variance explained by the principal components. The dashed line marks the 30 components retained.</em></p>

### Functions

Data check, cleaning, EDA pipeline and PCA feature engineering. The pipeline below calls these functions.

In [ ]:
META_COLS = ["County", "Station_Name", "Observation_ID"]


#=============================================helper functions=============================================
def _clean_meta(df):
    """create a copy dataframe from the raw"""

    df = df.copy()
    df["County"] = df["County"].astype(str).str.strip().str.lower()
    df["Station_Name"] = df["Station_Name"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True) #using the regex (statsprogramming) to avoid "Male", "male" etc.
    df["Observation_ID"] = df["Observation_ID"].astype(str).str.strip().str.lower()
    return df


def _to_matrix(df, name):
    """create matrix [stations x dates] from raw data"""

    #copy
    df = _clean_meta(df)

    #remove duplicates and keep track
    n_dup = df.duplicated("Observation_ID").sum()
    df = df.drop_duplicates("Observation_ID", keep="first").set_index("Observation_ID")
    meta = df[["County", "Station_Name"]]
    vals = df.drop(columns=["County", "Station_Name"])

    #column names "05/10/2014"
    vals.columns = pd.to_datetime(vals.columns, format="%d/%m/%Y")
    print(f"df_{name}: {n_dup} duplicates removed")
    return vals, meta


def _robust_z(X):
    """robust z-score per station (row): (x - median) / (1.4826 * MAD)"""

    med = X.median(axis=1)
    mad = X.sub(med, axis=0).abs().median(axis=1) * 1.4826
    return X.sub(med, axis=0).div(mad.replace(0, np.nan), axis=0)

def _undersample(avg, mn, mx, meta, min_coverage=0.9):
    """cut every station to the SAME period"""

    first = pd.to_datetime(avg.apply(lambda r: r.first_valid_index(), axis=1))
    last  = pd.to_datetime(avg.apply(lambda r: r.last_valid_index(),  axis=1))
    start, end = first.median(), last.median()
    coverage = avg.loc[:, start:end].notna().mean(axis=1)
    keep = coverage.index[coverage >= min_coverage]
    drop = coverage.index[coverage < min_coverage]

    print(f"UNDERSAMPLING: common period {start.date()} -> {end.date()} ({avg.loc[:, start:end].shape[1]} dates)")
    print(f"  {len(keep)} stations kept (>= {min_coverage:.0%} coverage), {len(drop)} dropped: "
          f"{meta.loc[drop, 'Station_Name'].tolist()[:5]}{' ...' if len(drop) > 5 else ''}")

    avg, mn, mx = (d.loc[keep, start:end] for d in (avg, mn, mx))
    return avg, mn, mx, meta.loc[keep]

def balance_check(X, names, title, save=None):
    """measured days per station: balanced = every station has the sam amount of data in the same perio"""

    X = X.copy()
    X.columns = pd.to_datetime(X.columns, format="%d/%m/%Y")      #raw data has text column names
    n_obs = X.notna().sum(axis=1)                                 #measured days per station
    first = X.apply(lambda r: r.first_valid_index(), axis=1)      #start per station
    last  = X.apply(lambda r: r.last_valid_index(),  axis=1)      #end per station
    D = X.shape[1]

    print(f"BALANCE CHECK ({title}): measured days per station")
    for label, sid in [("fewest", n_obs.idxmin()), ("most", n_obs.idxmax())]:
        print(f"  {label:7s}: {n_obs[sid]:5d}/{D} days  {first[sid].date()} -> {last[sid].date()}  ({names[sid]})")
    print(f"  median : {n_obs.median():5.0f}/{D} days")
    print(f"  starts : {first.min().date()} ... {first.max().date()}")
    print(f"  ends   : {last.min().date()} ... {last.max().date()}")
    overlap = first.max() <= last.min()
    print("  period covered by ALL stations: "
          + (f"{first.max().date()} -> {last.min().date()}" if overlap else "none -> data is IMBALANCED"))

    #plotting helped by ChatGPT
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(np.arange(len(n_obs)), n_obs.to_numpy(), step="mid", color="#2a78d6", linewidth=0)
    ax.axhline(D, color="#52514e", linestyle="--", linewidth=1)                 #maximum possible: all dates
    ax.text(0, D, f" all {D} dates", va="bottom", color="#52514e")
    ax.set(title=f"{title}: measured days per station - balanced data would be one flat block at the dashed line",
           xlabel=f"station (file order, n = {len(n_obs)})", ylabel="measured days",
           xlim=(0, len(n_obs) - 1), ylim=(0, D * 1.1))
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    if save:                                            #saved for the report text, not shown twice
        fig.savefig(save, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


#=============================================raw data check=============================================
def check_raw_data(df_avg, df_min, df_max, df_locs):
    """does not change any data but i want to check the raw data before i clean it"""

    print("============START CHECKING THE RAW DATA ============")

    frames = {"avg": df_avg, "min": df_min, "max": df_max}

    #Question: empty data, ID
    for name, df in frames.items():
        assert not df.empty, f"df_{name} is empty"#the file is not empty
        assert set(META_COLS).issubset(df.columns), f"df_{name} misses meta columns" #the three meta columns exist
        date_cols = df.columns.drop(META_COLS)
        assert pd.to_datetime(date_cols, format="%d/%m/%Y", errors="coerce").notna().all(), f"df_{name}: not all columns are dates"
        assert df["Observation_ID"].notna().all(), f"df_{name}: missing Observation_ID"    #all rows have an ID

    #Question : all four files contain exactly the same stations
    assert len({frozenset(d["Observation_ID"]) for d in (df_avg, df_min, df_max, df_locs)}) == 1, "files contain different stations"

    #Question: duplicates & the NaN
    for name, df in {**frames, "locs": df_locs}.items():
        values = df.drop(columns=META_COLS)
        print(f"df_{name:5s}: shape {df.shape}, "
              f"{df['Observation_ID'].duplicated().sum()} duplicate IDs, "
              f"{values.isna().sum().sum()}/{values.size} values missing, "
              f"{(values.dtypes == object).sum()} text columns")

    #numeric matrices [stations x dates] for the next two checks
    va, vn, vx = (df.set_index("Observation_ID").drop(columns=["County", "Station_Name"]).apply(pd.to_numeric, errors="coerce") for df in (df_avg, df_min, df_max))

    #Question: balanced data
    balance_check(va, df_avg.set_index("Observation_ID")["Station_Name"], "RAW DATA", save="Figure1.png")

    #Question : reasonable physics: min <= avg <= max must hold
    vn, vx = vn.reindex_like(va), vx.reindex_like(va)
    n_viol = ((vn > va + 0.006) | (va > vx + 0.006)).sum().sum()
    print(f"min <= avg <= max violated in {n_viol} cells")


#=============================================data cleaning=============================================
def data_cleaning(df_avg, df_min, df_max, df_locs, min_coverage=0.9, outlier_z=8.0, spike_z=50.0):
    """clean data with the points observed from data raw checking"""

    print("============CLEANING THE DATA============")

    #use helper to create the matrix structure
    avg, meta = _to_matrix(df_avg, "avg")
    mn, _ = _to_matrix(df_min, "min")
    mx, _ = _to_matrix(df_max, "max")

    #NEW: measurement errors -> values > spike_z robust std from the station median are impossible -> NaN
    #done for every file separately (min and max have their own spikes) and BEFORE undersampling
    for name, d in (("avg", avg), ("min", mn), ("max", mx)):
        spikes = _robust_z(d).abs() > spike_z
        d[spikes] = np.nan
        print(f"df_{name}: {spikes.sum().sum()} impossible values (spikes) set to NaN")

    #undersampling to avoid unbalanced data
    avg, mn, mx, meta = _undersample(avg, mn, mx, meta, min_coverage)

    #remove duplicates & stations with missing / impossible coordinates
    locs = _clean_meta(df_locs)
    n_dup = locs.duplicated("Observation_ID").sum()
    locs = locs.drop_duplicates("Observation_ID").set_index("Observation_ID")[["Latitude", "Longitude"]]
    valid = locs["Latitude"].between(-90, 90) & locs["Longitude"].between(-180, 180)
    locs = locs[valid]
    print(f"df_locs: {n_dup} duplicates removed, {(~valid).sum()} INVALID coordinates dropped")

    meta = meta.join(locs)

    #missing values: drop dates where no station measured anything
    empty = avg.columns[avg.isna().all(axis=0)]
    avg, mn, mx = (d.drop(columns=empty) for d in (avg, mn, mx))
    print(f"{len(empty)} empty dates dropped, {avg.isna().sum().sum()}/{avg.size} values missing")

    #outliers: only flag (> outlier_z robust std from the median) -> mostly real floods, kept
    outlier_mask = _robust_z(avg).abs() > outlier_z
    print(f"{outlier_mask.sum().sum()} values flagged as outliers (kept)")

    return avg, mn, mx, meta, outlier_mask


def make_comb_master(avg, mn, mx, meta, outlier_mask, complete_cases=True):
    """create master and comb file"""

    print("============CREATE MASTER AND COMB============")

    #comb: stack the three matrices next to each other, then attach meta
    comb = (pd.concat({"avg": avg.stack(), "min": mn.stack(), "max": mx.stack(),
                       "is_outlier": outlier_mask.stack()}, axis=1)
              .rename_axis(["Observation_ID", "Date"])
              .reset_index()
              .merge(meta.reset_index(), on="Observation_ID", how="left"))

    #master: avg directly, meta as row index (meta comes from avg, so same order)
    master = avg.copy()
    master.index = pd.MultiIndex.from_frame(meta.reset_index())
    if complete_cases:                       #PCA needs a full matrix -> drop dates with any NaN
        master = master.dropna(axis=1, how="any")
    return comb, master


#=============================================plot style=============================================
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, GRID = "#52514e", "#e5e4e0"


def style(ax, title, xlabel, ylabel):
    """take always the same style for the plots"""
    ax.set_title(title, loc="left", fontsize=11)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


#=============================================EDA pipeline=============================================
def eda_pipeline(path_avg="avg_river_levels.csv", path_min="min_river_levels.csv",
                 path_max="max_river_levels.csv", path_locs="monitoring_locations.csv", min_coverage=0.9):
    """load -> check raw data -> clean -> check balance again -> build comb and master"""

    #load data
    df_avg, df_min, df_max = pd.read_csv(path_avg), pd.read_csv(path_min), pd.read_csv(path_max)
    df_locs = pd.read_csv(path_locs)

    #check the raw data before cleaning
    check_raw_data(df_avg, df_min, df_max, df_locs)

    #clean all dataframes
    avg, mn, mx, meta, outlier_mask = data_cleaning(df_avg, df_min, df_max, df_locs, min_coverage=min_coverage)

    #balance check again: after undersampling every station covers the same period
    balance_check(avg, meta["Station_Name"], "after cleaning", save="Figure1b.png")

    #combined (long) and master (wide, PCA-ready) dataframe
    comb, master = make_comb_master(avg, mn, mx, meta, outlier_mask, complete_cases=True)
    display(master.head())
    display(comb.head())

    return avg, mn, mx, meta, outlier_mask, comb, master


#=============================================feature engineering: standardise + PCA=============================================
def standardise_rows(X):
    """z-score every station over time (row): compare WHEN levels rise and fall, not river size"""
    X = np.asarray(X, dtype=float)
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

row_scaler = FunctionTransformer(standardise_rows)      #StandardScaler would scale per day (column), not per station


def pca_features(master, n_pc=30):
    """cumulative explained variance -> PCA with n_pc components, returns the stations in PCA space"""

    #calculate the explained (cum) variance
    cum = np.cumsum(PCA().fit(standardise_rows(master)).explained_variance_ratio_)
    for n in [2, 5, 10, 20, 30, 50]:
        if n <= len(cum):
            print(f"{n:3d} PCs: {cum[n - 1]:.1%}")

    #plot: cumulative explained variance with the chosen number of PCs
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(np.arange(1, len(cum) + 1), cum, color=COLORS[0], linewidth=2)
    ax.axvline(n_pc, color=INK, linestyle="--", linewidth=1)
    ax.text(n_pc + 3, 0.05, f"{n_pc} PCs = {cum[n_pc - 1]:.0%}", color=INK)
    ax.set_ylim(0, 1.02)
    style(ax, "Cumulative explained variance", "number of principal components", "cumulative share")
    plt.tight_layout()
    fig.savefig("Figure3.png", dpi=200, bbox_inches="tight")     #saved for the report text, not shown twice
    plt.close(fig)

    #PCA pipeline: standardise every station -> keep n_pc components
    pca_pipe = make_pipeline(row_scaler, PCA(n_components=n_pc))
    Z = pca_pipe.fit_transform(master)
    print(f"Z: {Z.shape[0]} stations x {Z.shape[1]} principal components")
    return Z, pca_pipe


### Pipeline

Load, check and clean the data (Figure 1), then standardise every station and reduce to 30 principal components.

In [ ]:
avg, mn, mx, meta, outlier_mask, comb, master = eda_pipeline(min_coverage=0.9)
Z, pca_pipe = pca_features(master, n_pc=30)

============START CHECKING THE RAW DATA ============
df_avg  : shape (392, 4382), 0 duplicate IDs, 525877/1716568 values missing, 0 text columns
df_min  : shape (392, 4382), 0 duplicate IDs, 525877/1716568 values missing, 0 text columns
df_max  : shape (392, 4382), 0 duplicate IDs, 525877/1716568 values missing, 0 text columns
df_locs : shape (392, 5), 0 duplicate IDs, 0/784 values missing, 0 text columns
BALANCE CHECK (RAW DATA): measured days per station
  fewest :   452/4379 days  2022-05-14 -> 2024-07-16  (Elliot Water, Kelly Den - Arbilot)
  most   :  4373/4379 days  2014-10-05 -> 2026-10-08  (River Tweed, Sprouston)
  median :  3283/4379 days
  starts : 2014-10-05 ... 2024-11-30
  ends   : 2021-11-15 ... 2026-10-08
  period covered by ALL stations: none -> data is IMBALANCED
min <= avg <= max violated in 0 cells
============CLEANING THE DATA============
df_avg: 0 duplicates removed
df_min: 0 duplicates removed
df_max: 0 duplicates removed
df_avg: 140 impossible values (spikes) se

,,,,,2017-07-08,2017-07-09,2017-07-10,2017-07-31,2017-08-01,2017-08-02,2017-08-03,2017-08-07,2017-08-25,2017-08-26,...,2026-08-20,2026-08-21,2026-08-23,2026-08-24,2026-08-25,2026-08-26,2026-08-27,2026-08-28,2026-08-29,2026-08-30
Observation_ID,County,Station_Name,Latitude,Longitude,,,,,,,,,,,,,,,,,,,,,
aberdeenshire__bogie-redcraig__obs1,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015,0.181,0.173,0.170,0.155,0.171,0.157,0.156,0.155,0.179,0.173,...,0.153,0.173,0.131,0.120,0.120,0.115,0.110,0.121,0.142,0.135
aberdeenshire__dee-grampian-mar-lodge__obs1,aberdeenshire,"Dee (Grampian), Mar Lodge",56.988040,-3.486198,0.649,0.641,0.652,0.683,0.698,0.674,0.673,0.760,0.739,0.777,...,0.739,0.754,0.657,0.635,0.619,0.608,0.599,0.591,0.606,0.609
aberdeenshire__dee-grampian-park__obs1,aberdeenshire,"Dee (Grampian), Park",57.075424,-2.335825,0.837,0.808,0.857,0.836,0.831,0.829,0.824,0.878,0.941,0.891,...,0.814,0.835,0.794,0.767,0.752,0.741,0.731,0.727,0.727,0.727
aberdeenshire__dee-grampian-polhollick__obs1,aberdeenshire,"Dee (Grampian), Polhollick",57.055146,-3.083181,0.421,0.401,0.414,0.440,0.466,0.438,0.427,0.496,0.500,0.516,...,0.451,0.464,0.396,0.374,0.361,0.352,0.343,0.338,0.342,0.346
aberdeenshire__dee-grampian-woodend__obs1,aberdeenshire,"Dee (Grampian), Woodend",57.050129,-2.603567,0.525,0.495,0.510,0.526,0.538,0.522,0.511,0.576,0.624,0.593,...,0.481,0.493,0.451,0.425,0.408,0.400,0.388,0.380,0.380,0.388


,Observation_ID,Date,avg,min,max,is_outlier,County,Station_Name,Latitude,Longitude
0,aberdeenshire__bogie-redcraig__obs1,2017-03-31,0.221,0.219,0.223,False,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015
1,aberdeenshire__bogie-redcraig__obs1,2017-04-01,0.221,0.216,0.231,False,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015
2,aberdeenshire__bogie-redcraig__obs1,2017-04-02,0.220,0.214,0.228,False,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015
3,aberdeenshire__bogie-redcraig__obs1,2017-04-03,0.212,0.209,0.214,False,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015
4,aberdeenshire__bogie-redcraig__obs1,2017-04-04,0.208,0.204,0.211,False,aberdeenshire,"Bogie, Redcraig",57.424758,-2.802015


  2 PCs: 29.8%
  5 PCs: 50.9%
 10 PCs: 65.2%
 20 PCs: 76.8%
 30 PCs: 82.7%
 50 PCs: 89.5%
Z: 301 stations x 30 principal components


## Regression / Classification

*(supervised part of the group)*

# Model Fitting and Tuning
(to be deleted after all task:

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. All displayed figures should be described and referenced in the text.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

)

## Clustering

### Model choice

We cluster the stations with **k-means**, applied to the 30 principal components of the standardised water level series (see Feature Engineering). K-means groups stations by their distance in this space, so stations whose levels rise and fall at the same times end up together. We chose k-means because it is simple, scales easily to our data, and produces cluster centres that can be read as a "typical" water level pattern per cluster. We considered clustering on per-station summary statistics (mean, standard deviation, skewness) instead. We rejected this because these statistics are dominated by river size and were heavily distorted by the measurement errors described above, whereas the full time series captures *when* rivers respond. The complete model is a single sklearn pipeline: per-station standardisation → PCA (30 components) → k-means.

### Number of clusters

The elbow curve (Figure 3, left panel) decreases smoothly without a clear kink, and the silhouette scores are low for all k (0.17–0.25, Figure 3, right panel).

<p align="center">
  <img src="Figure4.png" width="30%">
</p>
<p align="center"><em>Figure 3: Elbow curve (left) and silhouette score (right) of k-means for k = 2 to 10.</em></p>

This indicates that the stations form a continuum rather than sharply separated groups. Since no single criterion is decisive, we compare candidate values of k on three criteria (Table 1): separation (silhouette), stability (mean adjusted Rand index, ARI, between five random seeds with 100 initialisations each; 1 means identical solutions), and the size of the smallest cluster.

<div align="center">

| k | Silhouette | Stability (ARI) | Smallest cluster |
|---|---|---|---|
| **4** | **0.221** | **1.000** | **30** |
| 5 | 0.220 | 0.954 | 29 |
| 6 | 0.239 | 0.915 | 14 |
| 7 | 0.246 | 0.937 | 14 |
| 8 | 0.252 | 0.898 | 14 |

*Table 1: Comparison of candidate numbers of clusters.*

</div>

Only k = 4 and k = 5 avoid very small clusters; for every k ≥ 6, a group of only 14 stations is split off. Between k = 4 and k = 5 the silhouette scores are virtually identical, and k = 4 is perfectly reproducible (ARI = 1.00 vs. 0.95), so we choose **k = 4**. Larger k increase the silhouette only slightly, and only by separating this small group.

In [ ]:
#=============================================NUMBER OF CLUSTERS=============================================
ks = range(2, 11)
fits = [KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=0).fit(Z) for k in ks]

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ks, [f.inertia_ for f in fits], color=COLORS[0], marker="o")
style(ax[0], "Elbow", "k", "inertia")
ax[1].plot(ks, [silhouette_score(Z, f.labels_) for f in fits], color=COLORS[0], marker="o")
style(ax[1], "Silhouette", "k", "silhouette")
plt.tight_layout()
fig.savefig("Figure4.png", dpi=200, bbox_inches="tight")
plt.close(fig)

#compare candidate k
rows = []
for k in [4, 5, 6, 7, 8]:
    runs = [KMeans(n_clusters=k, init="k-means++", n_init=100, random_state=s).fit(Z) for s in range(5)]
    best_run = min(runs, key=lambda km: km.inertia_)
    rows.append(dict(k=k,
                     silhouette=silhouette_score(Z, best_run.labels_),
                     stability=np.mean([adjusted_rand_score(best_run.labels_, r.labels_) for r in runs]),
                     smallest_cluster=np.bincount(best_run.labels_).min()))
display(pd.DataFrame(rows).set_index("k").round(3))

,silhouette,stability,smallest_cluster
k,,,
4,0.221,1.000,30
5,0.220,0.954,29
6,0.239,0.915,14
7,0.246,0.937,14
8,0.252,0.898,14


### K-means++: initialisation and final model

K-means only finds a local optimum, and the result depends on the initial cluster centres. We use the **k-means++** initialisation, which spreads the initial centres apart, and examine how the solution depends on the random seed (random_state) and the number of restarts (n_init; the best of n_init runs is kept) (Figure 4).

<p align="center">
  <img src="Figure5.png" width="60%">
</p>
<p align="center"><em>Figure 4: K-means++ for different random seeds (rows) and numbers of restarts n_init (columns). Grey: stations before assignment; black crosses: cluster centres.</em></p>

In [ ]:
#=============================================SIMULATE THE ASSIGNMENT (WORKSHOP)=============================================x

K_FINAL = 4

random_states = [0, 1, 2, 3]
n_inits = [1, 2, 5, 10, 20]
results = []

#one figure: one row per random_state, first column = stations before clustering
fig, axes = plt.subplots(len(random_states), len(n_inits) + 1, figsize=(19, 3 * len(random_states)),
                         sharex=True, sharey=True, squeeze=False)
for row, rs in zip(axes, random_states):

    #first panel: stations before clustering (nothing assigned yet)
    row[0].scatter(Z[:, 0], Z[:, 1], s=6, color=INK, alpha=0.6, edgecolor="none")
    style(row[0], f"random_state = {rs}: not assigned", "PC1", "PC2")

    #other panels: k-means++ with increasing number of restarts
    for a, n in zip(row[1:], n_inits):
        km = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=n, random_state=rs).fit(Z)
        results.append(dict(random_state=rs, n_init=n, inertia=km.inertia_))

        a.scatter(Z[:, 0], Z[:, 1], s=6, c=[COLORS[l] for l in km.labels_], edgecolor="none")             #stations
        a.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], s=60, c="black", marker="X")       #centres
        style(a, f"n_init={n}, inertia={km.inertia_:.0f}", "PC1", "")
plt.tight_layout()
fig.savefig("Figure5.png", dpi=200, bbox_inches="tight")     #saved for the report text, not shown twice
plt.close(fig)

#inertia of all runs as one table: rows = random_state, columns = n_init
display(pd.DataFrame(results).pivot(index="random_state", columns="n_init", values="inertia").round(1))

#=============================================K-MEANS++=============================================
#final model: n_init=100 for every random_state -> does the seed still matter?
finals = [KMeans(n_clusters=K_FINAL, init="k-means++", n_init=100, random_state=rs).fit(Z) for rs in random_states]
for rs, km in zip(random_states, finals):
    print(f"random_state={rs}: inertia={km.inertia_:.1f}")

#keep the run with the lowest inertia (= what n_init does internally)
best = min(finals, key=lambda km: km.inertia_)
labels = best.labels_
print("agreement with best run (ARI):", [round(adjusted_rand_score(labels, km.labels_), 2) for km in finals])

clusters = master.index.to_frame(index=False).assign(cluster=labels)    #cluster per station
print(f"stations per cluster: {np.bincount(labels).tolist()}")

#final result: usage of ChatGPT for this one :)
fig = plt.figure(figsize=(16, 5))
ax0 = fig.add_subplot(1, 3, 1)
ax1 = fig.add_subplot(1, 3, 2)
ax2 = fig.add_subplot(1, 3, 3, projection=ccrs.PlateCarree())

#map background for (c): land, sea, coastline
ax2.set_extent([-8, -0.5, 54.5, 61], crs=ccrs.PlateCarree())
ax2.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#f2f1ee")
ax2.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="#e6eef6")
ax2.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.5, edgecolor=INK)

for c in range(K_FINAL):
    m = labels == c
    kw = dict(s=15, color=COLORS[c], edgecolor="none", label=f"cluster {c} (n={m.sum()})")
    ax0.scatter(Z[m, 0], Z[m, 1], **kw)
    ax1.scatter(clusters.loc[m, "Longitude"], clusters.loc[m, "Latitude"], **kw)
    ax2.scatter(clusters.loc[m, "Longitude"], clusters.loc[m, "Latitude"], transform=ccrs.PlateCarree(), **kw)

style(ax0, "(a) Clusters in PCA space", "PC1", "PC2")
style(ax1, "(b) Clusters by coordinates", "Longitude", "Latitude")
ax1.set_aspect(1 / np.cos(np.deg2rad(clusters["Latitude"].mean())))   #no distorted map
ax2.set_title("(c) Clusters on the map of Scotland", loc="left", fontsize=11)
ax2.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
fig.savefig("Figure6_map.png", dpi=200, bbox_inches="tight")
plt.close(fig)


n_init,1,2,5,10,20
random_state,,,,,
0,85186.3,85186.3,84263.4,84263.4,84263.4
1,84285.6,84285.6,84285.6,84263.4,84263.4
2,84891.1,84891.1,84657.9,84657.9,84296.1
3,92712.1,84293.0,84263.4,84263.4,84263.4


random_state=0: inertia=84263.4
random_state=1: inertia=84263.4
random_state=2: inertia=84263.4
random_state=3: inertia=84263.4
agreement with best run (ARI): [1.0, 1.0, 1.0, 1.0]
stations per cluster: [30, 80, 129, 62]


With a single initialisation, the result depends on the seed: the within-cluster sum of squares (inertia) ranges from 84,283 to 92,709, and the cluster assignments differ visibly (Figure 4). With ten restarts, three of the four seeds reach the same lowest inertia (84,261.3); the remaining seed still ends in a slightly worse solution even with 20 restarts. For the final model we therefore use **100 restarts**. With this setting, all four seeds converge to the identical solution (inertia 84,261.3, pairwise ARI = 1.00), so the result no longer depends on the random initialisation.

The final clusters contain 30, 80, 129 and 62 stations (Figure 5). Although the clustering used no location information, the clusters are clearly **spatially coherent** on the map. This suggests that neighbouring rivers respond to the same weather systems. The clusters are characterised in the next section.

<p align="center">
  <img src="Figure6_map.png" width="40%">
</p>
<p align="center"><em>Figure 5: Final clusters (k = 4) (a) in the space of the first two principal components, (b) by station coordinates and (c) on a map of Scotland (ChatGPT).</em></p>

## Regression / Classification

*(supervised part of the group)*

# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final clustering and of your final supervised learning model, its performance, and reliability. You should discuss what the implications of your models are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the monitoring of river levels but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may influence the classification of flood events OR the prediction of future water levels (depending on which question you are answering), which may be useful for the government officials to improve their understanding of floods or water level forecasting, and potentially assist in the development of effective river management and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

In [ ]:
#=============================================CLUSTER PROFILES=============================================
#all features are computed on the cleaned daily levels and do not depend on the gauge's reference level
lab = clusters.set_index("Observation_ID")["cluster"]                 #cluster per station
A = avg.loc[lab.index]                                                #cleaned daily levels [stations x days]
Zs = A.sub(A.mean(axis=1), axis=0).div(A.std(axis=1), axis=0)        #standardised per station (as in the PCA)
month = A.columns.month

prof = pd.DataFrame({
    "cluster":        lab,
    "Latitude":       meta.loc[lab.index, "Latitude"],
    "Longitude":      meta.loc[lab.index, "Longitude"],
    "skewness":       A.skew(axis=1),                                             #few extreme peaks -> large
    "autocorr_1d":    A.apply(lambda s: s.autocorr(lag=1), axis=1),               #low = flashy, high = slow river
    "seasonality":    Zs.loc[:, month.isin([12, 1, 2])].mean(axis=1)              #winter minus summer level,
                    - Zs.loc[:, month.isin([6, 7, 8])].mean(axis=1),              #in standard deviations
    "flood_days_yr":  outlier_mask.loc[lab.index].sum(axis=1)                     #flagged extreme days per year
                    / (A.notna().sum(axis=1) / 365.25),
})

#table: one row per cluster (median over its stations), plus size and most common counties
summary = prof.groupby("cluster").median().round(2)
summary.insert(0, "n_stations", prof.groupby("cluster").size())
summary["main_counties"] = (meta.loc[lab.index, "County"].groupby(lab)
                            .agg(lambda s: ", ".join(s.value_counts().index[:3])))
display(summary)

#plot: (a) seasonal cycle per cluster, (b) typical response during one winter
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
monthly = Zs.T.groupby(month).mean().T.groupby(lab).mean()           #cluster x month
winter = Zs.loc[:, "2025-10-01":"2026-03-31"].groupby(lab).mean()     #cluster x day
for c in summary.index:
    ax[0].plot(monthly.columns, monthly.loc[c], color=COLORS[c], marker="o", linewidth=2, label=f"cluster {c}")
    ax[1].plot(winter.columns, winter.loc[c], color=COLORS[c], linewidth=1, label=f"cluster {c}")
style(ax[0], "(a) Seasonal cycle", "month", "mean standardised level")
ax[0].set_xticks(range(1, 13))
style(ax[1], "(b) Winter 2025/26", "date", "mean standardised level")
ax[1].legend(frameon=False, fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
fig.savefig("Figure7_profiles.png", dpi=200, bbox_inches="tight")
plt.close(fig)

,n_stations,Latitude,Longitude,skewness,autocorr_1d,seasonality,flood_days_yr,main_counties
cluster,,,,,,,,
0,30,56.62,-4.91,0.43,0.57,0.25,1.61,"wigtown, argyll-and-bute, ayrshire-and-arran"
1,80,56.55,-2.82,2.17,0.86,1.22,0.94,"aberdeenshire, angus, perth-and-kinross"
2,129,55.70,-3.97,1.50,0.81,1.10,0.11,"perth-and-kinross, dumfries, ayrshire-and-arran"
3,62,57.56,-4.39,1.28,0.81,1.08,0.06,"inverness, ross-and-cromarty, sutherland"


## Clustering

The 301 stations fall into four groups of rivers that rise and fall at the same times (Figure 5). Although the clustering used only the water level series and no location information, each group occupies a distinct part of Scotland. This shows that the timing of high and low water is driven mainly by regional weather rather than by individual river characteristics. Figure 6 compares the typical seasonal cycle of each group and their behaviour during winter 2025/26, and Table 2 summarises their main properties.

<p align="center">
  <img src="Figure7_profiles.png" width="40%">
</p>
<p align="center"><em>Figure 6: Typical water level of each cluster (standardised per station) (a) by calendar month and (b) during winter 2025/26.</em></p>

<div align="center">

| Cluster | Stations | Main counties | Seasonality (SD) | Day-to-day persistence | Skewness | Extreme days / year |
|---|---|---|---|---|---|---|
| 0 | 30 | Wigtown, Argyll & Bute, Ayrshire & Arran | 0.25 | 0.57 | 0.43 | 1.61 |
| 1 | 80 | Aberdeenshire, Angus, Perth & Kinross | 1.22 | 0.86 | 2.17 | 0.94 |
| 2 | 129 | Perth & Kinross, Dumfries, Ayrshire & Arran | 1.10 | 0.81 | 1.50 | 0.11 |
| 3 | 62 | Inverness, Ross & Cromarty, Sutherland | 1.08 | 0.81 | 1.28 | 0.06 |

*Table 2: Median properties of the stations in each cluster. Seasonality: winter minus summer level in standard deviations. Day-to-day persistence: correlation between consecutive days (1 = very slow changes). Extreme days: days more than 8 robust standard deviations from the station's median, in either direction.*

</div>

**Cluster 2: central and southern Scotland (129 stations).** The largest group covers the Central Belt and the south (Figure 5c). Levels follow a strong seasonal cycle (1.1 standard deviations higher in winter than in summer), rising early in autumn and reaching their lowest point in May. In winter they respond to most rain events with moderate peaks (Figure 6b), but extreme levels are rare (0.11 days per year). These are the "typical" Scottish rivers of our data.

**Cluster 3: northern Highlands (62 stations).** Located mainly in Inverness, Ross & Cromarty and Sutherland, these rivers have almost the same properties as cluster 2: similar seasonality, persistence and skewness, and the fewest extreme days (0.06 per year). They differ from cluster 2 mainly in *which* storms affect them, which makes them the northern counterpart of cluster 2.

**Cluster 1: eastern Scotland (80 stations).** Rivers in Aberdeenshire, Angus and Perth & Kinross have the strongest seasonal contrast (1.22), with their highest levels in December. They change most slowly from day to day (persistence 0.86) and rise latest in autumn and fall slowest in spring. At the same time, they have the most pronounced high-water episodes: the highest skewness (2.17) and almost one extreme day per year. In winter 2025/26 they stayed high for several weeks in late January and February while the other groups had already returned to normal (Figure 6b). This points to large, slow-responding catchments, possibly supported by snowmelt from the eastern Highlands. For flood management, the risk here is prolonged rather than sudden high water.

**Cluster 0: atypical coastal stations (30 stations).** This group lies mainly along the western and south-western coast (Wigtown, Argyll & Bute, Ayrshire & Arran), with single stations on the islands and Shetland. Its behaviour differs clearly from the other groups: almost no seasonal cycle (0.25), strong day-to-day fluctuations (persistence 0.57), and the most extreme days (1.61 per year, high and low). It barely follows the major events that dominate the other groups. This pattern is consistent with gauges influenced by tides, lakes or very local rainfall, but our data cannot confirm the cause. These stations should be treated individually rather than as one river type.

# Generative AI statement

For Clustering usage of ChatGPT mainly for Plot generation (especially the Map plot) Then using ChatGPT and Claude when it comes to the prints with function f{} helped me very well.

# References

*Include a list of all references cited in the text.*

In [ ]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb

[NbConvertApp] Converting notebook project.ipynb to pdf
[NbConvertApp] Support files will be in project_files/
[NbConvertApp] Making directory ./project_files
[NbConvertApp] Writing 120929 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] CRITICAL | xelatex failed: ['xelatex', 'notebook.tex', '-quiet']
b"This is XeTeX, Version 3.141592653-2.6-0.999998 (TeX Live 2026) (preloaded format=xelatex)\n restricted \\write18 enabled.\nentering extended mode\n(./notebook.tex\nLaTeX2e <2026-06-01>\nL3 programming layer <2026-09-09>\n(/Users/pierrickdeboer/Library/TinyTeX/texmf-dist/tex/latex/base/article.cls\nDocument Class: article 2025/01/22 v1.4n Standard LaTeX document class\n(/Users/pierrickdeboer/Library/TinyTeX/texmf-dist/tex/latex/base/size11.clo))\n\n! LaTeX Error: File `tcolorbox.sty' not found.\n\nType X to quit or <RETURN> to proceed,\nor enter new name. (Default extension: sty)\n\nEnter file 